# Phase 1 : 데이터 적재 및 전처리 파이프라인

## 1. 라이브러리 임포트

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from datetime import datetime

print("Polars version:", pl.__version__)

Polars version: 2.0.0


## 2. 데이터 로드

In [2]:
file_path = "./데이터/data.json"  # 본인의 데이터 파일명으로 변경

df_lazy = pl.scan_ndjson(file_path)

# 상위 5개 데이터 확인
print(df_lazy.head(5).collect())

# 전체 데이터 스키마 확인
print(df_lazy.collect_schema())

shape: (5, 18)
┌────────┬────────┬────────────┬───────────┬───┬──────────────┬─────────────┬─────────────┬────────┐
│ status ┆ gender ┆ length     ┆ firstName ┆ … ┆ userAgent    ┆ song        ┆ artist      ┆ method │
│ ---    ┆ ---    ┆ ---        ┆ ---       ┆   ┆ ---          ┆ ---         ┆ ---         ┆ ---    │
│ i64    ┆ str    ┆ f64        ┆ str       ┆   ┆ str          ┆ str         ┆ str         ┆ str    │
╞════════╪════════╪════════════╪═══════════╪═══╪══════════════╪═════════════╪═════════════╪════════╡
│ 200    ┆ M      ┆ 524.32934  ┆ Shlok     ┆ … ┆ "Mozilla/5.0 ┆ Ich mache   ┆ Popol Vuh   ┆ PUT    │
│        ┆        ┆            ┆           ┆   ┆ (Windows NT  ┆ einen       ┆             ┆        │
│        ┆        ┆            ┆           ┆   ┆ 6.1; …       ┆ Spiegel -   ┆             ┆        │
│        ┆        ┆            ┆           ┆   ┆              ┆ Drea…       ┆             ┆        │
│ 200    ┆ F      ┆ 238.39302  ┆ Vianney   ┆ … ┆ "Mozilla/5.0 ┆ MiÃÂ©ntele 

## 3. 데이터 정렬 및 시간 변환
### 밀리초 단위의 타임스탬프를 보기 편한 Datetime으로 변환하고, 유저별 시간 순서대로 정렬

In [3]:
df = (
    df_lazy
    .with_columns([
        # 밀리초 타임스탬프를 Datetime으로 변환
        pl.from_epoch("ts", time_unit="ms").alias("datetime"),
        pl.from_epoch("registration", time_unit="ms").alias("reg_datetime")
    ])
    .sort(["userId", "ts"])  # 유저별, 시간순 정렬
    .collect()
)

print(f"총 레코드 수: {df.height:,} 건, 총 유저 수: {df['userId'].n_unique():,} 명")

총 레코드 수: 26,259,199 건, 총 유저 수: 22,278 명


## 이탈(Churn) 타겟 라벨 생성
### 'Cancellation Confirmation' 페이지를 방문한 기록이 있는 유저를 이탈(1), 아니면 유지(0)로 정의

In [4]:
# 이탈 유저 ID 집합 추출
churned_users = (
    df.filter(pl.col("page") == "Cancellation Confirmation")
    .select("userId")
    .unique()
    .to_series()
    .to_list()
)

# 전체 유저 테이블에 이탈 라벨(is_churn) 부여
df_labeled = df.with_columns(
    pl.when(pl.col("userId").is_in(churned_users))
    .then(1)
    .otherwise(0)
    .alias("is_churn")
)

# 유저별 이탈 분포 확인
user_churn_summary = df_labeled.select(["userId", "is_churn"]).unique()
print(user_churn_summary["is_churn"].value_counts())

shape: (2, 2)
┌──────────┬───────┐
│ is_churn ┆ count │
│ ---      ┆ ---   │
│ i32      ┆ u32   │
╞══════════╪═══════╡
│ 1        ┆ 5003  │
│ 0        ┆ 17275 │
└──────────┴───────┘


# 5. 전처리 완료된 데이터 Parquet 저장
# 이후 피처 엔지니어링에서 빠르게 불러올 수 있도록 저장

In [6]:
output_path = "./데이터/processed_data.parquet"
df_labeled.write_parquet(output_path)

print(f"Phase 1 완료! 파일이 성공적으로 저장되었습니다: {output_path}")

Phase 1 완료! 파일이 성공적으로 저장되었습니다: ./데이터/processed_data.parquet
